# Retrieval Augmented Generation (RAG) improves LLM responses by retrieving relevant information before generation.
Traditional RAG relies on vector similarity, which struggles with:

* multi-hop reasoning

* relational queries

* concept-to-concept navigation

GraphRAG solves this by:

* extracting entities & relations

* storing them as a graph

* retrieving information via semantic + graph traversal

#What This Notebook Contains

* Creation of a small document corpus

* Triple extraction (subject–relation–object)

* Graph construction using **NetworkX**

* Semantic retrieval using sentence embeddings

* Graph expansion (multi-hop retrieval)

* Conversion of subgraph into LLM-ready context

* Clear observations on what each step does

#Step 1: Install Dependencies

In [ ]:
!pip -q install networkx sentence-transformers numpy matplotlib


#Step 2: Create 4 Short Documents

In [ ]:
docs2 = [
    "Machine learning is a subset of artificial intelligence that learns patterns from data.",

    "Deep learning uses neural networks with many layers to model complex patterns.",

    "Neural networks are inspired by the human brain and consist of interconnected neurons.",

    "Artificial intelligence is used in education for personalized learning systems."
]


In [ ]:
def normalize(text):
    return text.strip().lower()


#Step 3: Extract Knowledge Triples

In [ ]:
import re


In [ ]:
def extract_triples(text):
    triples = []
    patterns = [
        (r"(.+?) stands for (.+?)\.", "stands_for"),
        (r"(.+?) is a subset of (.+?)\.", "is_subset_of"),
        (r"(.+?) uses (.+?)\.", "uses"),
        (r"(.+?) are inspired by (.+?)\.", "inspired_by"),
        (r"(.+?) consist[s]? of (.+?)\.", "consists_of"),
        (r"(.+?) used in (.+?)\.", "used_in"),
    ]

    for pat, rel in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            s, o = [normalize(g) for g in m.groups()]
            triples.append((s, rel, o))

    return triples


#Step 4: Build the Knowledge Graph

In [ ]:
import networkx as nx

G2 = nx.DiGraph()

for doc in docs2:
    triples2 = extract_triples(doc)
    for h, r, t in triples2:
        G2.add_node(h)
        G2.add_node(t)
        G2.add_edge(h, t, relation=r, source=doc)

print("Nodes:", G2.number_of_nodes())
print("Edges:", G2.number_of_edges())
list(G2.edges(data=True))


#Step 5: Embed Graph Nodes (Semantic Layer)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

nodes2 = list(G2.nodes())
node_embeddings2 = model.encode(nodes2, convert_to_numpy=True)


#Step 6: Graph-based Retrieval

In [ ]:
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def graph_retrieve(query, top_nodes=2, hops=2):
    q_emb = model.encode([query], convert_to_numpy=True)[0]

    sims = [cosine_sim(q_emb, emb) for emb in node_embeddings2]
    top_idx = np.argsort(sims)[::-1][:top_nodes]
    seed_nodes = [nodes2[i] for i in top_idx]

    expanded = set(seed_nodes)

    for _ in range(hops):
        new_nodes = set()
        for n in expanded:
            if n in G2:
                new_nodes.update(G2.successors(n))
                new_nodes.update(G2.predecessors(n))
        expanded.update(new_nodes)

    subG = G2.subgraph(expanded).copy()
    return seed_nodes, subG


#Step 7: Convert Subgraph → Context

In [ ]:
def subgraph_to_context(subG):
    lines = []
    for u, v, data in subG.edges(data=True):
        lines.append(f"{u} --[{data['relation']}]--> {v}")
    return "\n".join(lines)


#Step 8: Run a Query (Multi-hop!)

In [ ]:
query = "How are neural networks related to artificial intelligence in education?"


seed_nodes, subG = graph_retrieve(query)

print("Seed Nodes:", seed_nodes)
print("\nRetrieved Context:\n")
print(subgraph_to_context(subG))


#Step 9: Visualize the Retrieved Graph

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
pos = nx.spring_layout(subG, seed=42)
nx.draw(subG, pos, with_labels=True, node_size=2500)
edge_labels = nx.get_edge_attributes(subG, "relation")
nx.draw_networkx_edge_labels(subG, pos, edge_labels=edge_labels)
plt.show()


# Observations

- Semantic similarity selects initial seed nodes based on meaning.
- Graph expansion retrieves related concepts through relations.
- Multi-hop retrieval connects AI → ML → Neural Networks → Education.
- Compared to vector-only RAG, GraphRAG preserves relational context.
